Pixelization: Source Science
============================

Source science focuses on studying the highly magnified properties of the background lensed source galaxy (or galaxies).

Using the reconstructed source pixelization, we can compute key quantities such as the magnification, total flux, and
intrinsic size of the source.

For pixelized source reconstructions, these calculations can be quite involved as they required speciifc code to
handle irregular mesh pixels and other quantities. We illustrate how to perform these calculations below.

However, this does make the source reconstructions different to share with other people, as it would mean they need
to understand how to manipulate irregular meshes. The end of this example shows how a .csv source reconstruction file
is output by a pixelization model-fit, which allows anyone to easy interpolate the source reconstruction on to a uniform grid
for analysis without the need for PyAutoLens.

__Contents__

- **Model Fit:** Perform the model-fit using the search and analysis.
- **Extra Galaxies Noise Scaling:** Scale the faint extra galaxy out of the fit so its light is not counted as source.
- **Interpolated Source:** The simplest way to perform source science calculations on a pixelized source reconstruction is to.
- **Source Flux:** A key quantity for a source galaxy is its total flux, which can be used to compute magnitudes (see.
- **Zoom:** The interpolation grid above was large in extent (-3.0" to 3.0" in both the y and x directions).
- **Errors:** The interpolated errors on the source reconstruction can also be computed, which will allow you to.
- **Magnification:** The overall magnification of the source is estimated as the ratio of total surface brightness in.
- **Masking:** Reconstructions can be imperfect, for example having faint source flux in pixels at the edge of the.
- **Magnification via Mesh:** The magnification above used an interpolation of the source reconstruction onto a uniform grid.
- **True Magnification:** The simulator's true magnification, to check the reconstruction against.
- **Magnification At A Point:** The signed magnification `mu = 1 / det(A)` at chosen (y,x) image-plane coordinates, computed via the Hessian.
- **Reconstruction CSV:** In the results `image` folder there is a .csv file called `source_plane_reconstruction_0.csv` which.

__Google Colab Setup__

Set up Google Colab as in [the introductory notebook](https://github.com/PyAutoLabs/autolens_workspace/blob/main/start_here.ipynb).


In [ ]:
# @title Set up Google Colab { display-mode: "form" }
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

from autolens import setup_notebook; setup_notebook()

import numpy as np
from pathlib import Path
import autofit as af
import autolens as al
import autolens.plot as aplt

__Model Fit__

The code below is identical to the pixelizaiton `modeling` example, crucially creating a model-fit which
outputs the pixelization source reconstruction to a .csv file.

In [ ]:
dataset_name = "simple__no_lens_light"
dataset_path = Path("dataset") / "imaging" / dataset_name

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if al.util.dataset.should_simulate(str(dataset_path)):
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "scripts/imaging/features/no_lens_light/simulator.py"],
        check=True,
    )

# Intentional raw guard: positions.json is a file (should_simulate rmtree's a directory); the
# should_simulate guard above already deletes the whole dataset folder under PYAUTO_SMALL_DATASETS,
# so this file-level check re-fires whenever the dataset is regenerated.
if not (dataset_path / "positions.json").exists():
    import subprocess
    import sys

    subprocess.run(
        [
            sys.executable,
            "scripts/imaging/data_preparation/examples/optional/positions.py",
        ],
        check=True,
    )

dataset = al.Imaging.from_fits(
    data_path=dataset_path / "data.fits",
    psf_path=dataset_path / "psf.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    pixel_scales=0.1,
)

__Extra Galaxies Noise Scaling__

The dataset contains a faint extra galaxy inside the mask. Its light is not part of the lensed source, but a
pixelized source would reconstruct it as source structure and count it in the fluxes and magnifications below.
We therefore scale it out of the fit with the `mask_extra_galaxies.fits` shipped with the dataset, as
`imaging/modeling.py` explains.

In [ ]:
mask_extra_galaxies = al.Mask2D.from_fits(
    file_path=dataset_path / "mask_extra_galaxies.fits",
    pixel_scales=dataset.pixel_scales,
    invert=True,
)

dataset = dataset.apply_noise_scaling(mask=mask_extra_galaxies)

mask_radius = 3.0

mask = al.Mask2D.circular(
    shape_native=dataset.shape_native,
    pixel_scales=dataset.pixel_scales,
    radius=mask_radius,
)

dataset = dataset.apply_mask(mask=mask)

dataset = dataset.apply_over_sampling(
    over_sample_size_pixelization=4,
)

lens_galaxy = al.Galaxy(
    redshift=0.5,
    mass=al.mp.Isothermal(
        centre=(0.0, 0.0),
        einstein_radius=1.6,
        ell_comps=al.convert.ell_comps_from(axis_ratio=0.9, angle=45.0),
    ),
)

field = al.MassField(
    redshift=0.5,
    shear=al.mp.ExternalShear(gamma_1=0.05, gamma_2=0.05),
)

mesh_pixels_yx = 28
mesh_shape = (mesh_pixels_yx, mesh_pixels_yx)

mesh = al.mesh.RectangularBilinearAdaptDensity(shape=mesh_shape)
regularization = al.reg.Constant(coefficient=1.0)

pixelization = al.Pixelization(mesh=mesh, regularization=regularization)

source_galaxy = al.Galaxy(redshift=1.0, pixelization=pixelization)

tracer = al.Tracer(galaxies=[lens_galaxy, source_galaxy], fields=[field])

fit = al.FitImaging(
    dataset=dataset,
    tracer=tracer,
)

inversion = fit.inversion

mapper = inversion.cls_list_from(cls=al.Mapper)[
    0
]  # Extract the mapper from the inversion


We plot the fit, confirming that the pixelized source reconstruction provides a good fit to the data.

Note how the pixelized source reconstruction is performed on an irregular adaptive grid of rectangular pixels,
which is denser in regions of high magnification. This non-uniform distribution of pixels means we need to be care
when performing source science calculations, especially a quantity like the magnification which depends on area.

In [ ]:
aplt.subplot_fit_imaging(fit=fit)

All information about the pixelized source reconstruction is contained in the `Inversion` object, which can be
accessed via `fit.inversion`.

In [ ]:
inversion = fit.inversion
print(f"Inversion Object: {inversion}")

For example, the reconstructed source pixel flux values are stored in the `reconstruction` attribute of the inversion.

In [ ]:
reconstruction = inversion.reconstruction

print(f"Reconstructed Source Pixel Fluxes: {reconstruction}")

total_flux = np.sum(reconstruction)

print(f"Total Source Flux via Pixelization: {total_flux} e- s^-1")

In order to perform source science calculations we need to know which flux value corresponds to which pixel in the 
source-plane.

This information is available in the inversion, below we print the (y,x) centre of each source pixel corresponding to 
the `reconstruction` values printed above.

In [ ]:
mapper = inversion.cls_list_from(cls=al.Mapper)[
    0
]  # Extract the mapper from the inversion

source_plane_mesh_grid = mapper.source_plane_mesh_grid

print(f"Source Plane Mesh Grid Coordinates: {source_plane_mesh_grid}")

The image-plane reconstruction can also be computed from the inversion, which is called 
the `mapped_reconstructed_operated_data` and as seen above is needed to compute the magnification.

In [ ]:
mapped_reconstructed_operated_data = inversion.mapped_reconstructed_operated_data

print(f"Mapped Reconstructed Image: {mapped_reconstructed_operated_data}")

__Interpolated Source__

The simplest way to perform source science calculations on a pixelized source reconstruction is to interpolate
its values to a uniform 2D grid of pixels, which can therefore be stored using a `Array2D` object,
which is basically just a 2D numpy array (see the `Data Structure` section at the top of this example).

We interpolate the rectangular pixelized source reconstruction to a new uniform grid we call the `interpolation_grid`.
This calculation can be quite slow, so to make this example run fast we use a relatively small grid, but in practice
you may wish to use a larger grid (e.g. 100x1000 pixels or larger for actual science calculations).

Interpolation grid points outside the mesh have no reconstructed source light, so `fill_value=0.0` sets them
to zero rather than `NaN`, which would otherwise make every sum below `NaN`.

In [ ]:
from scipy.interpolate import griddata

interpolation_grid = al.Grid2D.uniform(shape_native=(200, 200), pixel_scales=0.05)

interpolated_reconstruction = griddata(
    points=source_plane_mesh_grid,
    values=reconstruction,
    xi=interpolation_grid,
    fill_value=0.0,
)

# As a pure 2D numpy array in case its useful for calculations
interpolated_reconstruction_ndarray = interpolated_reconstruction.reshape(
    interpolation_grid.shape_native
)

interpolated_reconstruction = al.Array2D.no_mask(
    values=interpolated_reconstruction_ndarray,
    pixel_scales=interpolation_grid.pixel_scales,
)

By printing the interpolated array, we confirm it is a 2D array and can see the pixel values of the source 
reconstruction.

We also plot the interpolated source reconstruction using an `aplt.plot_array`.

In [ ]:
print(interpolated_reconstruction.native)

aplt.plot_array(array=interpolated_reconstruction, title="")

__Source Flux__

A key quantity for a source galaxy is its total flux, which can be used to compute magnitudes (see 
`autolens_workspace/*/guides/units/flux`) example for more details on this).

The total flux of the source reconstruction can now be computed by summing the interpolated array.

The units of the light profile `intensity` are the units of the data the light profile was fitted to. In this example
we will assume everything is in electrons per second (`e- s^-1`), which is typical for Hubble Space Telescope imaging data.

In [ ]:
total_source_flux = np.sum(interpolated_reconstruction)

print(f"Total Source Flux via Interpolated Pixelization: {total_source_flux} e- s^-1")

__Zoom__

The interpolation grid above was large in extent (-3.0" to 3.0" in both the y and x directions), meaning that
the source was a small flux was a small region of this grid.

By changing the `extent` of the interpolation grid, we can performed the interpolation zoomed in on only the
regions of the source-plane where the source reconstruction has non-negligible flux. This
makes the interpolation more accurate, as the interpolation ican use more pixels in the region of interest,
and also makes visualizing the source reconstruction easier.

In [ ]:
extent = (-1.0, 1.0, -1.0, 1.0)
shape_native = (401, 401)

interpolation_grid_zoom = al.Grid2D.from_extent(
    extent=extent,
    shape_native=shape_native,
)

interpolated_reconstruction = griddata(
    points=source_plane_mesh_grid,
    values=reconstruction,
    xi=interpolation_grid_zoom,
    fill_value=0.0,
)


# As a pure 2D numpy array in case its useful for calculations
interpolated_reconstruction_ndarray = interpolated_reconstruction.reshape(
    interpolation_grid_zoom.shape_native
)

interpolated_reconstruction = al.Array2D.no_mask(
    values=interpolated_reconstruction_ndarray,
    pixel_scales=interpolation_grid_zoom.pixel_scales,
)

__Errors__

The interpolated errors on the source reconstruction can also be computed, which will allow you to perform
model-fitting of the source reconstruction.

Note that some source-plane pixels have a noise value of `NaN`. These are the pixels the inversion never
solved for: the mesh zeroes its poorly-constrained edge pixels before solving, to stop them absorbing flux
and destabilizing the linear algebra, and their reconstruction is therefore an exact `0.0` rather than a
fitted value. A pixel with no fitted value has no uncertainty either, so the noise map reports `NaN` --
"never estimated" -- rather than a number that would look like a real error bar.

This is not something to work around; it is the reconstruction and the noise map agreeing about which
pixels were actually fitted.

Use `np.isnan(reconstruction_noise_map)` to find these pixels, and not `reconstruction == 0.0`. The two are
not the same set, and the difference is large. The inversion's non-negative solver also pins pixels it *did*
solve for at exactly `0.0` -- wherever the fit wants no flux -- and those are real fitted values with real
error bars. On a typical compact-source fit with a 28x28 mesh, 603 of the 784 source pixels read `0.0`,
while only the 108 edge pixels the mesh zeroed are `NaN`. A `NaN` noise value always means the
reconstruction is `0.0` there; a reconstruction of `0.0` does not mean the pixel was never fitted.

`NaN` propagates through interpolation, so the interpolated noise map below is blank in the region covered
by those edge pixels -- which is honest, since there is nothing there to interpolate from. If you need an
interpolated error map that covers the full field, drop the `NaN` entries from `points` and `values` before
calling `griddata` rather than filling them with a number.

In [ ]:
reconstruction_noise_map = inversion.reconstruction_noise_map

interpolated_noise_map = griddata(
    points=source_plane_mesh_grid,
    values=reconstruction_noise_map,
    xi=interpolation_grid,
)

# As a pure 2D numpy array in case its useful for calculations
interpolated_noise_map_ndarray = interpolated_noise_map.reshape(
    interpolation_grid.shape_native
)

interpolated_noise_map = al.Array2D.no_mask(
    values=interpolated_noise_map_ndarray, pixel_scales=interpolation_grid.pixel_scales
)

aplt.plot_array(array=interpolated_noise_map, title="")

__Magnification__

The overall magnification of the source is estimated as the ratio of total surface brightness in the image-plane and 
total surface brightness in the source-plane.

Note that the surface brightness is different to the total flux above, as surface brightness is flux per unit area. 
We therefore explicitly mention how area folds into the calculation below.

The interpolated source reconstruction above has different sized pixels in the image-plane and source-plane, so 
we need to explicitly account for area when computing the magnification.

The `pixel_area` attribute of the `Array2D` object gives us the area of each pixel in arcseconds squared, which we
can use to compute the magnification below.

In [ ]:
magnification = np.sum(
    mapped_reconstructed_operated_data * mapped_reconstructed_operated_data.pixel_area
) / np.sum(interpolated_reconstruction * interpolated_reconstruction.pixel_area)

print(f"Magnification via Interpolated Source: {magnification}")

__Masking__

Reconstructions can be imperfect, for example having faint source flux in pixels at the edge of the
source-plane that through comparison to the data are not a genuine part of the source. This can impact
the calculation of the source flux and magnification.

If you want to be extra careful, you can use a mask to zero the source-plane pixels that you do not trust and use
this to remove pixels from source science calculations.

Another approach, which we use below, is we create a source-plane signal-to-noise map and use this to create a mask 
that removes all pixels with a signal-to-noise < 5.0.

The `NaN` noise values described above pass through this calculation safely, and it is worth knowing why.
For an edge pixel the division is `0.0 / NaN`, which is `NaN` -- not a divide-by-zero, and it raises no
warning. `NaN < 5.0` then evaluates to `False`, so those pixels are *not* selected by `mesh_pixel_mask`
and keep their reconstruction value, which is already `0.0`. They therefore contribute nothing to the
source flux and magnification computed below, which is the correct outcome: a pixel that was never fitted
should neither be counted as significant nor be able to move the result.

In [ ]:
signal_to_noise_map = reconstruction / reconstruction_noise_map

mesh_pixel_mask = signal_to_noise_map < 5.0

reconstruction_masked = reconstruction.copy()
reconstruction_masked[mesh_pixel_mask] = 0.0

interpolated_reconstruction_masked = griddata(
    points=source_plane_mesh_grid,
    values=reconstruction_masked,
    xi=interpolation_grid,
    fill_value=0.0,
)

# As a pure 2D numpy array in case its useful for calculations
interpolated_reconstruction_masked_ndarray = interpolated_reconstruction_masked.reshape(
    interpolation_grid.shape_native
)

interpolated_reconstruction_masked = al.Array2D.no_mask(
    values=interpolated_reconstruction_masked_ndarray,
    pixel_scales=interpolation_grid.pixel_scales,
)

aplt.plot_array(array=interpolated_reconstruction_masked, title="")

__Magnification via Mesh__

The magnification above used an interpolation of the source reconstruction onto a uniform grid. The magnification
can also be computed directly on the mesh of the source reconstruction, without an interpolation grid.

The reconstruction is a value at every mesh pixel, but the source it describes is continuous: between the pixels its
surface brightness is filled in by interpolation. Each pixel's value therefore spreads over a small patch of the
source plane, weighted by a function that is 1 at the pixel and falls to 0 at its neighbours (its basis function).
The source plane area that belongs to a pixel is the integral of that function. Multiplying every pixel's value by
its area and summing gives the total surface brightness of the interpolated source, which is the denominator of the
magnification.

These areas are given by the `areas_for_magnification` attribute of the mapper's mesh geometry. How they are
computed depends on the mesh:

- `RectangularBilinearAdaptDensity`, used in this example, places its pixels on a grid that is regular in stretched
  coordinates, which pack pixels more closely where more image pixels trace to in the source plane. Between pixels
  the source is interpolated bilinearly, so each basis function is a pyramid in the stretched coordinates. Its area
  is computed exactly by integrating it along the y and x axes through the stretching.
- `RectangularUniform` has square pixels of equal size, so every pixel's area is the area of one pixel.
- `Delaunay` interpolates linearly across triangles, so a pixel's area is exactly one third of the total area of the
  triangles that have it as a corner.
- `KNearestNeighbor` and `KNNBarycentric` integrate their own interpolation weights numerically, by evaluating them
  on a fine grid that covers the mesh.
- `DelaunayNN` uses the `Delaunay` areas, which approximate its natural-neighbour interpolation to within about
  0.5%.

On simulated data whose true magnification is known and whose source is well resolved by the mesh, this
calculation recovers the true magnification to within about 1% for nearly every mesh, and within 2% for all of
them. Any remaining difference comes from the source reconstruction itself, not from the areas, as the
`__True Magnification__` section below shows for this example's compact source.

The mesh calculation integrates the reconstruction with exactly the interpolation the fit used, so it does not
depend on the resolution or extent of an interpolation grid. The interpolation grid above instead joins the pixel
centres with straight lines, a different interpolation that can give a somewhat different total: in these examples
the two agree to within a few percent for imaging data, but differ much more for the interferometer example, whose
reconstruction varies strongly between neighbouring pixels. Use the mesh calculation for the magnification, and the
interpolation grid for plotting the source and inspecting it pixel by pixel.

The same calculation is performed during a model-fit by the `magnification` latent variable, so its posterior can
be estimated alongside the lens model (see `autolens_workspace/*/guides/results/latent_variables.py`).

In [ ]:
mesh_areas = mapper.mesh_geometry.areas_for_magnification

magnification = np.sum(
    mapped_reconstructed_operated_data * mapped_reconstructed_operated_data.pixel_area
) / np.sum(reconstruction * mesh_areas)

print(f"Magnification via Mesh: {magnification}")

__True Magnification__

This dataset is simulated, so its true magnification is known: the simulator's tracer is stored in the dataset
folder as `tracer.json`. We compute it the same way as above, as the total lensed source light inside the mask
divided by the total light of the source itself, using finely sampled grids so the light profiles are integrated
accurately.

Comparing the two shows how accurate the reconstruction is. The mesh areas themselves are exact or close to it, so
any difference comes from the reconstruction: regularization smooths the source, spreading some of its light onto
neighbouring pixels, which makes the source look slightly larger and the magnification slightly smaller. For a real
lens the truth is unknown, and this comparison is not possible.

In [ ]:
true_tracer = al.from_json(file_path=dataset_path / "tracer.json")

image_grid = al.Grid2D.from_mask(mask=mask, over_sample_size=8)
lensed_source_image = true_tracer.image_2d_list_from(grid=image_grid)[-1]

source_grid = al.Grid2D.uniform(
    shape_native=(1000, 1000), pixel_scales=0.01, over_sample_size=4
)
source_image = sum(
    galaxy.image_2d_from(grid=source_grid) for galaxy in true_tracer.planes[-1]
)

true_magnification = np.sum(
    lensed_source_image * lensed_source_image.pixel_area
) / np.sum(source_image * source_image.pixel_area)

print(f"True Magnification: {true_magnification}")

__Magnification At A Point__

The magnifications above are the overall magnification of the source: a ratio of fluxes integrated over the
source's full light distribution. Lens modeling codes also commonly quote the magnification at a single (y,x)
coordinate in the image-plane, for example at a lensed image, a compact clump in an arc or a transient.

This is computed from the deflection angles of the lens model via the Hessian, which gives the lensing Jacobian `A`
at each coordinate, with the magnification `mu = 1 / det(A)`.

The point magnification is signed, and its sign is the parity of the lensed image at that coordinate: positive for
images at minima (or maxima) of the arrival-time surface and negative for saddle-point images, which are
mirror-reversed relative to the source. Its absolute value is how much a point-like source at that location is
magnified. It is therefore distinct from the source magnification computed above, which averages the magnification
over the extended source and is always positive. Close to a critical curve `det(A)` goes to zero, so the point
magnification becomes very large and changes quickly with the coordinate.

The point magnification depends only on the mass model, not on the source reconstruction, so a pixelized source
fit is used exactly as a parametric one. Below, we compute it at three (y,x) coordinates on the lensed arcs of the
simulated dataset.

For a real lens, pass the `max_log_likelihood_tracer` of your lens model and coordinates measured from your data.
For a multi-plane system (for example a cluster with sources at several redshifts), the magnification must be
computed up to the plane of the source of interest with `plane_j`; see the `Trap (b)` section of
`autolens_workspace/*/guides/advanced/multi_plane.py`.

In [ ]:
point_grid = al.Grid2DIrregular(values=[(0.3, 1.7), (-1.5, 0.2), (0.0, -1.7)])

lens_calc = al.LensCalc.from_tracer(tracer=tracer)

magnification_at_points = lens_calc.magnification_2d_via_hessian_from(grid=point_grid)

print(f"Magnification At Points (signed): {magnification_at_points}")
print(f"Magnification At Points (absolute): {abs(magnification_at_points)}")

The dataset folder also contains the `positions.json` file of the source's multiple image positions, which is used
for the positions likelihood when modeling. Evaluating the magnification at these positions gives the magnification
and parity of each multiple image.

In [ ]:
positions = al.Grid2DIrregular(
    al.from_json(file_path=Path(dataset_path, "positions.json"))
)

magnification_at_positions = lens_calc.magnification_2d_via_hessian_from(grid=positions)

print(f"Multiple Image Positions: {positions}")
print(f"Magnification At Multiple Images (signed): {magnification_at_positions}")

__Reconstruction CSV__

In the results `image` folder there is a .csv file called `source_plane_reconstruction_0.csv` which contains the
y and x coordinates of the pixelization mesh, the reconstruct values and the noise map of these values.

This file is provides all information on the source reconstruction in a format that does not depend autolens
and therefore be easily loaded to create images of the source or shared collaborations who do not have PyAutoLens
installed.

We now perform a lens model fit, which will create this .csv file in the modeling output folder.

First, lets load `source_plane_reconstruction_0.csv` as a dictionary, using basic `csv` functionality in Python.

In [ ]:
# Lens:

mass = af.Model(al.mp.PowerLaw)
lens = af.Model(al.Galaxy, redshift=0.5, mass=mass)

# External Shear:

field = af.Model(al.MassField, redshift=0.5, shear=af.Model(al.mp.ExternalShear))

# Source:
mesh = af.Model(al.mesh.RectangularBilinearAdaptDensity, shape=mesh_shape)
regularization = af.Model(al.reg.Constant)

pixelization = af.Model(al.Pixelization, mesh=mesh, regularization=regularization)

source = af.Model(al.Galaxy, redshift=1.0, pixelization=pixelization)

# Overall Lens Model:

model = af.Collection(
    galaxies=af.Collection(lens=lens, source=source),
    fields=field,
)

search = af.Nautilus(
    path_prefix=Path("features"),
    name="pixelization",
    unique_tag=dataset_name,
    n_live=100,
    n_batch=20,
    iterations_per_quick_update=50000,
)

positions = al.Grid2DIrregular(
    al.from_json(file_path=Path(dataset_path, "positions.json"))
)

positions_likelihood = al.PositionsLH(positions=positions, threshold=0.3)

analysis = al.AnalysisImaging(
    dataset=dataset,
    positions_likelihood_list=[positions_likelihood],
    use_jax=True,
)

result = search.fit(model=model, analysis=analysis)

__Reconstruction CSV__

The file ``source_plane_reconstruction_0.csv` provides all information on the source reconstruction in a format that 
does not depend autolens and therefore be easily loaded to create images of the source or shared collaborations who 
do not have PyAutoLens installed.

First, lets load `source_plane_reconstruction_0.csv` as a dictionary, using basic `csv` functionality in Python.

NOTE: If the .csv file does not exist, we create a dictionary with the same format but with dummy values so the rest of
the script can be run.

In [ ]:
import csv

try:

    with open(
        search.paths.image_path / "source_plane_reconstruction_0.csv", mode="r"
    ) as file:
        reader = csv.reader(file)
        header_list = next(reader)  # ['y', 'x', 'reconstruction', 'noise_map']

        reconstruction_dict = {header: [] for header in header_list}

        for row in reader:
            for key, value in zip(header_list, row):
                reconstruction_dict[key].append(float(value))

        # Convert lists to NumPy arrays
        for key in reconstruction_dict:
            reconstruction_dict[key] = np.array(reconstruction_dict[key])

except FileNotFoundError:

    print("`source_plane_reconstruction_0.csv` not found. Using dummy data instead.")

    x = np.array([-1.0, 0.0, 1.0, -1.0, 0.0, 1.0, -1.0, 0.0, 1.0])
    y = np.array([1.0, 1.0, 1.0, 0.0, 0.0, 0.0, -1.0, -1.0, -1.0])
    reconstruction = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0])
    noise_map = np.array([1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0])

    reconstruction_dict = {
        "x": x,
        "y": y,
        "reconstruction": reconstruction,
        "noise_map": noise_map,
    }

print(reconstruction_dict["y"])
print(reconstruction_dict["x"])
print(reconstruction_dict["reconstruction"])
print(reconstruction_dict["noise_map"])

You can now use standard libraries to performed calculations with the reconstruction on the mesh, again avoiding
the need to use autolens.

For example, we can create a RectangularBilinearAdaptDensity mesh using the scipy.spatial library, which is a triangulation
of the y and x coordinates of the pixelization mesh. This is useful for visualizing the pixelization
and performing calculations on the mesh.

In [ ]:
import scipy

points = np.stack(arrays=(reconstruction_dict["x"], reconstruction_dict["y"]), axis=-1)

mesh = scipy.spatial.Delaunay(points)

Interpolating the result to a uniform grid is also possible using the scipy.interpolate library, which means the result
can be turned into a uniform 2D image which can be useful to analyse the source with tools which require an uniform grid.

Below, we interpolate the result onto a 201 x 201 grid of pixels with the extent spanning -1.0" to 1.0", which
capture the majority of the source reconstruction without being too high resolution.

In [ ]:
from scipy.interpolate import griddata

values = reconstruction_dict["reconstruction"]

interpolation_grid = al.Grid2D.from_extent(
    extent=(-1.0, 1.0, -1.0, 1.0), shape_native=(201, 201)
)

interpolated_array = griddata(
    points=points, values=values, xi=interpolation_grid, fill_value=0.0
)
